# USDA VegScape & CROP-CASMA + Gridded Soil Moisture

**What these USDA products are.**
- **VegScape** (NASS) — daily/weekly **NDVI** and vegetation-condition indices from MODIS at
  250 m, 2000→present. Great for regional crop-condition monitoring.
- **CROP-CASMA** (NASS + NASA) — **soil moisture** from **SMAP** and vegetation from MODIS,
  packaged for agriculture.

Both are published primarily as **interactive web maps and OGC web services (WMS/WCS)** rather
than simple point JSON APIs — good for map layers, heavier for scripted point extraction.

**The practical path for Tracera.** For **high-res NDVI** use Sentinel-2 (`satellite_ndvi.ipynb`).
For **gridded soil moisture right now** — the gap our in-situ SCAN station couldn't fill — the
cleanest no-key source is **NASA POWER's land-model soil wetness** (`GWETTOP`/`GWETROOT`/
`GWETPROF`), which we pull below. Then we point you to VegScape/CROP-CASMA for the native
USDA MODIS/SMAP layers.

| | |
|---|---|
| **VegScape** | https://nassgeodata.gmu.edu/VegScape/ (WMS/WCS) |
| **CROP-CASMA** | https://cloud.csiss.gmu.edu/Croplandcros / NASS Crop-CASMA portal |
| **Soil moisture here** | NASA POWER — **free, no key** |

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Soil-moisture time series at the field (NASA POWER land model, 0–1 wetness)

In [2]:
POWER = "https://power.larc.nasa.gov/api/temporal/daily/point"

def power_soil_moisture(start, end, lat=LAT, lon=LON):
    params = ["GWETTOP", "GWETROOT", "GWETPROF", "PRECTOTCORR"]
    r = requests.get(POWER, params={"parameters": ",".join(params), "community": "AG",
        "latitude": lat, "longitude": lon, "start": start, "end": end, "format": "JSON"}, timeout=90)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    return df.replace(-999.0, pd.NA).rename(columns={
        "GWETTOP": "surface_wetness", "GWETROOT": "rootzone_wetness",
        "GWETPROF": "profile_wetness", "PRECTOTCORR": "precip_mm"})

sm = power_soil_moisture("20230601", "20230831").astype(float)
print("Surface vs root-zone soil moisture (0=dry, 1=saturated):")
sm.head()

Surface vs root-zone soil moisture (0=dry, 1=saturated):


,surface_wetness,rootzone_wetness,profile_wetness,precip_mm
2023-06-01,0.59,0.47,0.47,15.26
2023-06-02,0.52,0.47,0.47,1.16
2023-06-03,0.49,0.46,0.47,0.42
2023-06-04,0.47,0.46,0.47,0.56
2023-06-05,0.46,0.45,0.46,0.37


### 2. Interpret — dry-down between rains (surface reacts fastest)

In [3]:
summary = pd.DataFrame({
    "mean": sm.mean().round(3),
    "min": sm.min().round(3),
    "max": sm.max().round(3),
})
print(f"June–Aug 2023 total precip: {sm['precip_mm'].sum():.0f} mm")
print("Surface wetness swings most with rain/dry-down; profile is the buffer the crop draws on.")
summary

June–Aug 2023 total precip: 221 mm
Surface wetness swings most with rain/dry-down; profile is the buffer the crop draws on.


,mean,min,max
surface_wetness,0.407,0.34,0.59
rootzone_wetness,0.410,0.37,0.47
profile_wetness,0.419,0.38,0.47
precip_mm,2.406,0.00,32.58


### Notes & how to extend
- **Layers:** `GWETTOP` (~0–5 cm surface), `GWETROOT` (root zone), `GWETPROF` (full profile).
  Surface reacts to each rain; root-zone/profile track the water actually available to the crop.
- **Native USDA products:**
  - **VegScape** MODIS NDVI — pull via its **WMS/WCS** (`GetCoverage` for a bbox) for 250 m
    vegetation condition; good regional complement to Sentinel-2.
  - **CROP-CASMA** SMAP soil moisture (~1 km/9 km) — via the NASS Crop-CASMA portal / OGC
    services; higher-fidelity soil moisture than the land model when you need it.
  - **Raw SMAP** (L3/L4) is on NASA Earthdata and Google Earth Engine (`NASA_USDA/HSL/SMAP*`).
- Pair soil moisture with **OpenET** (crop water use) and **gridMET** (reference ET) for a full
  field water balance.